# f696原模型、接受目标与风险投影的beta饱和位置

只读封存源目标、原生Qlib账户和819个窗口，定位未受限却无beta余量的开盘日发生在哪一层。无新模型、动作或资金账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_f696_beta_saturation_layer_trace_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
a=root/s['action_card'];f=root/s['A0_card'];b=root/s['account_card']
for p,key in [(a/'summary.json','action_summary_sha256'),(a/'nas_readback.json','action_readback_sha256'),(f/'summary.json','A0_summary_sha256'),(b/'summary.json','account_summary_sha256'),(root/s['prior_headroom_plan'],'prior_headroom_plan_sha256'),(root/'research/stage1_f696_cash_to_soxl_legal_headroom_20261024/summary.json','prior_headroom_summary_sha256')]:assert sha(p)==s[key],key
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());ra=json.loads((a/'nas_readback.json').read_text())
sf=json.loads((f/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert ra['NAS_archive_sha256']==sa['nas_evidence']['sha256']==s['action_NAS_sha256'] and ra['artifact_hashes_checked']>=28
assert sf['nas_evidence']['sha256']==s['A0_NAS_sha256'] and sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==s['account_NAS_sha256']
for source,exp,rid,art,key in [(a,'action_experiment_id','action_recorder_id','inputs/raw_predictions','raw_predictions_sha256'),(a,'action_experiment_id','action_recorder_id','inputs/source_targets','source_targets_sha256'),(f,'A0_experiment_id','A0_recorder_id','inputs/frame','A0_frame_sha256'),(f,'A0_experiment_id','A0_recorder_id','native/A_active_research_replay_0','A0_native_sha256'),(b,'account_experiment_id','account_recorder_id','analysis/rolling126','account_rolling_sha256')]:
 assert sha(source/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['action_experiment_id'],recorder_id=s['action_recorder_id'])
 raw_dict=rec.load_object('inputs/raw_predictions');accepted_dict=rec.load_object('inputs/source_targets')
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['A0_experiment_id'],recorder_id=s['A0_recorder_id'])
 frame=rec.load_object('inputs/frame').reset_index(drop=True);native=rec.load_object('native/A_active_research_replay_0')
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['account_experiment_id'],recorder_id=s['account_recorder_id']);rolling=rec.load_object('analysis/rolling126')
assert len(frame)==len(rolling)+125==944
signal=pd.DatetimeIndex(frame.signal_date);trade=pd.DatetimeIndex(frame.trade_date)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))
assert signal.is_unique and signal.isin(calendar).all() and trade.is_unique
raw=raw_dict['frozen_current'].reindex(signal);accepted=accepted_dict['frozen_current'].reindex(signal)
columns=['soxl_weight','soxx_weight','cash_weight','tail_probability','alpha_probability']
assert list(raw.columns)==list(accepted.columns)==columns and raw.notna().all().all() and accepted.notna().all().all()
np.testing.assert_allclose(accepted[columns].to_numpy(float),frame[columns].to_numpy(float),rtol=0,atol=1e-7)
auth=native['authority'].reset_index(drop=True);targets=native['targets'].reset_index(drop=True)
assert pd.DatetimeIndex(auth.trade_date).equals(trade) and pd.DatetimeIndex(targets.trade_date).equals(trade)
execute=auth.execute_open_target.to_numpy(bool);cap=auth.current_beta_cap.to_numpy(float)
assert int(execute.sum())==797 and int((~execute).sum())==147
raw_beta=3*raw.soxl_weight.to_numpy(float)+raw.soxx_weight.to_numpy(float)
accepted_beta=3*accepted.soxl_weight.to_numpy(float)+accepted.soxx_weight.to_numpy(float)
projected_beta=3*targets.soxl_weight.to_numpy(float)+targets.soxx_weight.to_numpy(float)
assert np.isfinite(np.column_stack([raw_beta,accepted_beta,projected_beta,cap])).all()
assert (projected_beta<=cap+1e-8).all()
nonbinding=accepted_beta<=cap+1e-9
no_slack=cap-projected_beta<=1e-10
primary=execute&nonbinding&no_slack
assert int(primary.sum())==459
raw_high=raw_beta>=cap-1e-10
accepted_sat=accepted_beta>=cap-1e-10
project_created=accepted_beta<cap-1e-10
x=pd.DataFrame({'trade_date':trade,'year':trade.year,'quarter':trade.quarter,'execute':execute,'nonbinding':nonbinding,'no_slack':no_slack,'primary':primary,
                'raw_high':raw_high,'accepted_sat':accepted_sat,'project_created':project_created,
                'raw_beta':raw_beta,'accepted_beta':accepted_beta,'projected_beta':projected_beta,'cap':cap})
def cohort(g):
 p=g[g.primary];e=g[g.execute]
 return {'sessions':len(g),'execute':len(e),'nonbinding_execute':int((e.nonbinding).sum()),'primary_no_slack_nonbinding_execute':len(p),
         'primary_raw_model_beta_at_or_above_cap':int(p.raw_high.sum()),
         'primary_accepted_beta_at_cap':int(p.accepted_sat.sum()),
         'primary_accepted_below_cap_but_projection_at_cap':int(p.project_created.sum()),
         'mean_raw_minus_cap_primary':float((p.raw_beta-p.cap).mean()) if len(p) else None,
         'mean_accepted_minus_cap_primary':float((p.accepted_beta-p.cap).mean()) if len(p) else None}
all_days=cohort(x)
annual=[{'year':int(y),'partial_endpoint':y in (2023,2026),**cohort(g)} for y,g in x.groupby('year',sort=True)]
quarter=[{'year':int(y),'quarter':int(q),**cohort(g)} for (y,q),g in x.groupby(['year','quarter'],sort=True)]
assert len(annual)==4 and len(quarter)==16 and sum(v['sessions'] for v in annual)==944
windows=[]
for i in range(819):
 g=x.iloc[i:i+126];rw=rolling.iloc[i];assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 outcome='weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie'
 z=cohort(g)
 windows.append([str(g.trade_date.iloc[0].date()),outcome,z['primary_no_slack_nonbinding_execute'],z['primary_raw_model_beta_at_or_above_cap'],z['primary_accepted_beta_at_cap'],z['primary_accepted_below_cap_but_projection_at_cap']])
w=pd.DataFrame(windows,columns=['start','outcome','primary','raw_high','accepted_sat','projection_created'])
assert len(w)==819 and w.outcome.value_counts().to_dict()=={'weak':564,'good':255}
window_summary={name:{'windows':len(g),'mean_primary':float(g.primary.mean()),'mean_raw_high':float(g.raw_high.mean()),
                      'mean_accepted_sat':float(g.accepted_sat.mean()),'mean_projection_created':float(g.projection_created.mean())} for name,g in w.groupby('outcome')}
result={'status':'read_only_exact_f696_nonbinding_beta_saturation_layer_trace_no_policy','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'action':s['action_recorder_id'],'A0':s['A0_recorder_id'],'account':s['account_recorder_id']},
        'all_days':all_days,'annual':annual,'quarter':quarter,'rolling126_columns':list(w.columns),'rolling126':windows,'rolling126_summary':window_summary,
        'accepted_f696_max_abs_output_gap':float(np.max(np.abs(accepted[columns].to_numpy(float)-frame[columns].to_numpy(float)))),
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Raw model versus accepted target contrasts are source lineage, not causal feature effects','Cap saturation and weak/good windows are on the original own-state path and cannot define a policy','No fee-net candidate action, own-state account, 2/5bp or independent forward evidence']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'all_days':all_days,'annual':[(z['year'],z['primary_no_slack_nonbinding_execute'],z['primary_raw_model_beta_at_or_above_cap'],z['primary_accepted_beta_at_cap'],z['primary_accepted_below_cap_but_projection_at_cap']) for z in annual],
       'windows':window_summary})


{'all_days': {'sessions': 944, 'execute': 797, 'nonbinding_execute': 536, 'primary_no_slack_nonbinding_execute': 459, 'primary_raw_model_beta_at_or_above_cap': 243, 'primary_accepted_beta_at_cap': 459, 'primary_accepted_below_cap_but_projection_at_cap': 0, 'mean_raw_minus_cap_primary': 0.0004413932439125767, 'mean_accepted_minus_cap_primary': 0.0}, 'annual': [(2023, 139, 81, 139, 0), (2024, 155, 67, 155, 0), (2025, 111, 67, 111, 0), (2026, 54, 28, 54, 0)], 'windows': {'good': {'windows': 255, 'mean_primary': 67.43137254901961, 'mean_raw_high': 30.83137254901961, 'mean_accepted_sat': 67.43137254901961, 'mean_projection_created': 0.0}, 'weak': {'windows': 564, 'mean_primary': 59.969858156028366, 'mean_raw_high': 33.41843971631206, 'mean_accepted_sat': 59.969858156028366, 'mean_projection_created': 0.0}}}


## 中文结论

# f696 beta饱和发生在接受目标，风险投影并未额外制造

只读已封存的模型原始权重、EMA/HOLD接受目标、精确f696 A_0五项来源输出与0bp Tiered完整账户风险投影。944个信号日的接受输出与f696提案五项最大差 **0**；原生Qlib日历、来源SHA/NAS读回、797个EXECUTE/147个HOLD及819个连续126日SOXX相对窗口均核对。无拟合、动作、费用标签或新资金账户。

在原EXECUTE且接受目标 beta 不高于自身cap的536日中，**459日**后投影无beta余量。459日的**接受目标全部已等于cap**，不存在“接受目标低于cap、只因后续投影达到cap”的日子。原模型原始beta已达到/超过cap的有 **243日**；另 **216日**原模型低于cap，而EMA/HOLD接受目标达到cap。分年459日为2023/2024/2025/2026的 **139/155/111/54**，2023原始模型高beta为81日、其余58日在接受阶段达到cap。全部16季度与819个窗口逐窗计数见 `result.json`。

原账户未胜SOXX的564个滚动窗口，每窗平均这种“非超cap但已饱和”日 **59.97**；胜出255窗反而 **67.43**。所以不能将达到上限视为坏区间或自动放宽cap的交易开关。原模型输出与EMA/HOLD来源差异只是链路归因，不能证明平滑导致或修复了现金损失；完整账户的风险反馈、费用和SOXX门仍须重新计算。

决定 `read_only_accepted_f696_target_saturates_cap_before_projection_no_policy`。此前后投影现金转SOXL路线的未受限缺口，源于**进入投影前的接受配置目标已用尽beta容量**，不是b472投影凭空压掉剩余容量。下一机制若要同时改变配置意图与风险容量，必须有可在决策时区分有效现金保护和有价值cap释放的新信息，并在完整f696自身状态0bp账户上证明；不复跑已关闭静态旁路或用事后窗口选阈值。首次执行因计划中来源哈希抄录错误在Qlib对象读取前停止，`attempt1_preflight_failure.json`保留，机械修正未改公式或预算。SOXX v2未达成。
